# Comparison: Base GPT-2 vs DAPT Checkpoint

Loads a base GPT-2 model from OpenAI `.pkl` params, loads a DAPT model from a checkpoint created by `save_checkpoint()`, and compares token embedding vectors using cosine similarity; then looks at changes in next-token probabilities for representative texts.


## 1. Get directory paths

In [1]:
import os
import sys
import pickle
from pathlib import Path

import torch
import tiktoken

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "notebooks").exists():
            return p
    return start

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: /home/markb/cloned-llm-2026MAR16/llm-from-scratch
SRC_DIR: /home/markb/cloned-llm-2026MAR16/llm-from-scratch/src
DATA_DIR: /home/markb/cloned-llm-2026MAR16/llm-from-scratch/data


## 2. Import modules and establish tokenizer

In [2]:
from llm_from_scratch.models import gpt2
from llm_from_scratch.training import training_utils
from llm_from_scratch.configs import gpt2small_config
from llm_from_scratch.analysis import token_analysis as ta
from llm_from_scratch.analysis import weight_diff as wd

tokenizer = tiktoken.get_encoding("gpt2")
ta.tokenizer = tokenizer


## DEVICE/FILE PATH/FILE EXISTENCE  
Note that if DAPT file is not found, probably is in a subdirectory of /output:  
To fix:  
1. move to ./llm-from-scratch/data  
2. The file will also need to be renamed to "TEST_abstracts_epoch_lastsave_step_lastsave.pth"  

In [3]:
# Device selection: prefer CUDA if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Default file paths (edit if needed)
BASE_PARAMS_PATH = DATA_DIR / "gpt2_openai_params_124M.pkl"
DAPT_CKPT_PATH = DATA_DIR / "TEST_abstracts_epoch_lastsave_step_lastsave.pth"

print("BASE_PARAMS_PATH exists:", BASE_PARAMS_PATH.exists(), BASE_PARAMS_PATH)
print("DAPT_CKPT_PATH exists:", DAPT_CKPT_PATH.exists(), DAPT_CKPT_PATH)

if not BASE_PARAMS_PATH.exists():
    raise FileNotFoundError(f"Base params file not found: {BASE_PARAMS_PATH}")
if not DAPT_CKPT_PATH.exists():
    print("***File below is not found. Check in the output directory and subdirectory. Move to /data and rename to TEST_abstracts_epoch_lastsave_step_lastsave.pth")
    raise FileNotFoundError(f"DAPT checkpoint file not found: {DAPT_CKPT_PATH}")


Using device: cpu
BASE_PARAMS_PATH exists: True /home/markb/cloned-llm-2026MAR16/llm-from-scratch/data/gpt2_openai_params_124M.pkl
DAPT_CKPT_PATH exists: True /home/markb/cloned-llm-2026MAR16/llm-from-scratch/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth


## 3. Load base model from OpenAI .pkl params

In [4]:

base_cfg = dict(gpt2small_config.GPT_CONFIG_124M_OPENAI)
base_model = gpt2.setup_model(base_cfg).to(device)

with open(BASE_PARAMS_PATH, "rb") as f:
    base_params = pickle.load(f)

training_utils.load_weights_into_gpt(base_model, base_params)
base_model.eval()
print("Loaded base model from:", BASE_PARAMS_PATH)
print("Base tok_emb shape:", tuple(base_model.tok_emb.weight.shape))


Loaded base model from: /home/markb/cloned-llm-2026MAR16/llm-from-scratch/data/gpt2_openai_params_124M.pkl
Base tok_emb shape: (50257, 768)


In [5]:
# Inspect number of tensors total and in certain layers
base_named_params = list(base_model.named_parameters())
print("Number of named-parameter tensors in base_model:", len(base_named_params))

for block_idx in [0, 11]:
    block_prefix = f"trf_blocks.{block_idx}."
    block_param_names = [
        name for name, _ in base_named_params if name.startswith(block_prefix)
    ]
    print(f"\nNamed-parameter tensors in trf_blocks[{block_idx}] ({len(block_param_names)} total):")
    for name in block_param_names:
        print(name)


Number of named-parameter tensors in base_model: 197

Named-parameter tensors in trf_blocks[0] (16 total):
trf_blocks.0.att.W_query.weight
trf_blocks.0.att.W_query.bias
trf_blocks.0.att.W_key.weight
trf_blocks.0.att.W_key.bias
trf_blocks.0.att.W_value.weight
trf_blocks.0.att.W_value.bias
trf_blocks.0.att.out_proj.weight
trf_blocks.0.att.out_proj.bias
trf_blocks.0.ff.layers.0.weight
trf_blocks.0.ff.layers.0.bias
trf_blocks.0.ff.layers.2.weight
trf_blocks.0.ff.layers.2.bias
trf_blocks.0.norm1.scale
trf_blocks.0.norm1.shift
trf_blocks.0.norm2.scale
trf_blocks.0.norm2.shift

Named-parameter tensors in trf_blocks[11] (16 total):
trf_blocks.11.att.W_query.weight
trf_blocks.11.att.W_query.bias
trf_blocks.11.att.W_key.weight
trf_blocks.11.att.W_key.bias
trf_blocks.11.att.W_value.weight
trf_blocks.11.att.W_value.bias
trf_blocks.11.att.out_proj.weight
trf_blocks.11.att.out_proj.bias
trf_blocks.11.ff.layers.0.weight
trf_blocks.11.ff.layers.0.bias
trf_blocks.11.ff.layers.2.weight
trf_blocks.11.ff.

## 4. Load DAPT model from .pth checkpoint produced by save_checkpoint()
Note DAPT model was made by additional training with biomedical abstracts; see README.md

In [6]:

checkpoint = torch.load(DAPT_CKPT_PATH, map_location=device)

if "model_state_dict" not in checkpoint:
    raise KeyError("Checkpoint does not contain 'model_state_dict'.")

if "run_config" in checkpoint and "model_config" in checkpoint["run_config"]:
    dapt_model_cfg = checkpoint["run_config"]["model_config"]
else:
    # Fallback if run_config is missing
    dapt_model_cfg = base_cfg

dapt_model = gpt2.setup_model(dapt_model_cfg).to(device)
dapt_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
dapt_model.eval()

print("Loaded DAPT model from:", DAPT_CKPT_PATH)
print("DAPT tok_emb shape:", tuple(dapt_model.tok_emb.weight.shape))


Loaded DAPT model from: /home/markb/cloned-llm-2026MAR16/llm-from-scratch/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth
DAPT tok_emb shape: (50257, 768)


## 5. Model Difference Analysis Sections: Comparing the base (publically available weights) to the DAPT (additional training on biomedical abstracts) model


### 5a. Look at changes at the token embedding layer (+ output layer, as there is weight tying in this model)

In [7]:
# Extract token embedding matrices (base vs DAPT)
embed_initial = base_model.tok_emb.weight.detach().float().cpu().clone()
embed_after = dapt_model.tok_emb.weight.detach().float().cpu().clone()

print("shape_before:", tuple(embed_initial.shape))
print("shape_after:", tuple(embed_after.shape))


shape_before: (50257, 768)
shape_after: (50257, 768)


#### 5a.1 Do pairs of tokens that are closely associated in the biomedical text become closer in embedding in the DAPT model?

In [8]:
# Token-pair cosine similarity checks
import pandas as pd

words_dict = {
    "sp_HER2": tokenizer.encode(" HER2"),
    "HER_sp_2": tokenizer.encode("HER 2"),
    "sp_EGFR": tokenizer.encode(" EGFR"),
    "EG_sp_FR": tokenizer.encode("EG FR"),
    "ERBB2_only_BB2": tokenizer.encode("BB2"),
    "ERBB2_only_spERBB": tokenizer.encode(" ERBB"),
    "sp_kinase": tokenizer.encode(" kinase"),
    "cat vs dog": [tokenizer.encode(" cat")[0], tokenizer.encode(" dog")[0]],
}

resdf = pd.DataFrame(columns=[
    "word", "tokenid1", "token1", "tokenid2", "token2", "cos_sim_openai", "cos_sim_aftertrain", "ratio_afterVSbefore"
])

for word, (tokenid1, tokenid2) in words_dict.items():
    cos_sim_openai = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_initial)
    cos_sim_aftertrain = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_after)
    resdf.loc[len(resdf)] = {
        "word": word,
        "tokenid1": tokenid1,
        "token1": repr(tokenizer.decode([tokenid1])),
        "tokenid2": tokenid2,
        "token2": repr(tokenizer.decode([tokenid2])),
        "cos_sim_openai": cos_sim_openai,
        "cos_sim_aftertrain": cos_sim_aftertrain,
        "ratio_afterVSbefore": cos_sim_aftertrain / cos_sim_openai,
    }

print(resdf.to_string(index=False))


             word  tokenid1 token1  tokenid2 token2  cos_sim_openai  cos_sim_aftertrain  ratio_afterVSbefore
          sp_HER2     24906 ' HER'        17    '2'        0.259792            0.263318             1.013573
         HER_sp_2     16879  'HER'       362   ' 2'        0.171015            0.178956             1.046435
          sp_EGFR     41513  ' EG'     10913   'FR'        0.256762            0.250341             0.974991
         EG_sp_FR      7156   'EG'      8782  ' FR'        0.266662            0.237642             0.891174
   ERBB2_only_BB2     15199   'BB'        17    '2'        0.275225            0.272602             0.990468
ERBB2_only_spERBB     13793  ' ER'     15199   'BB'        0.255247            0.260255             1.019622
        sp_kinase     18967 ' kin'       589  'ase'        0.270267            0.271175             1.003361
       cat vs dog      3797 ' cat'      3290 ' dog'        0.549790            0.535333             0.973704


**RESULT**: Fairly small changes. ' HER' and '2' show a ~ +1% change. 'EG' and 'FR', which would be ' EGFR' show a slight decrease (~ -2%). The control of ' cat' and ' dog' show a small decrease (~ -2%). This implies that there may be larger changes in other parts of the model.

#### 5a.2 Do token embeddings stay basically the same across base model vs DAPT model? 

In [9]:
# Extract token embedding matrices and compute cosine similarity per token for base vs DAPT
base_tok_emb = base_model.tok_emb.weight.detach().float().cpu()
dapt_tok_emb = dapt_model.tok_emb.weight.detach().float().cpu()

if base_tok_emb.shape != dapt_tok_emb.shape:
    raise ValueError(
        f"Embedding shape mismatch: base={tuple(base_tok_emb.shape)} vs dapt={tuple(dapt_tok_emb.shape)}"
    )

cos_scores = ta.cosine_similarity_per_token(base_tok_emb, dapt_tok_emb)
print("cos_scores shape:", tuple(cos_scores.shape))
print("cosine min/max/mean:", cos_scores.min().item(), cos_scores.max().item(), cos_scores.mean().item())


cos_scores shape: (50257,)
cosine min/max/mean: 0.8841298818588257 0.9994819760322571 0.9706948399543762


In [10]:
# 4) Rank top 40 most changed and least changed tokens
TOP_K = 40

most_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_dissimilar",
)

least_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_similar",
)

print("Top 40 MOST changed tokens (lowest cosine):")
print(most_changed_df.to_string(index=False))

print()
print("Top 40 LEAST changed tokens (highest cosine):")
print(least_changed_df.to_string(index=False))


Top 40 MOST changed tokens (lowest cosine):
 tokenid          token  cosine_similarity
     921         ' You'           0.884130
    4705        ' Matt'           0.893998
    3497         ' Get'           0.899295
    5137     ' putting'           0.899304
    6889        ' Make'           0.899364
    1644      ' police'           0.899783
    6035         ' Dan'           0.900174
    2495      ' pretty'           0.900250
    5180       ' Chris'           0.900631
    1639          'You'           0.900823
    4995        ' Mike'           0.901138
    4422        ' Alex'           0.901382
   25508         ' 330'           0.901733
    1526         ' Mar'           0.902345
    5395         ' Jim'           0.902827
    2396           'So'           0.903508
    3932         ' Ben'           0.903555
    6209   ' basically'           0.903645
    7214        ' Take'           0.903873
    3807       ' movie'           0.904015
    1223   ' something'           0.904180
    1532  

**RESULT**: There are some changes and some appear significant, but not in the tokens that seem to be most associated with biomedical work

#### 5a.3 Do next token probabilities shift?

In [11]:

GPROMPTS = [
    " dogs and",
    "Many families like keeping animals in their homes - we call these pets. Among the most popular pets are dogs and",
    " HER",
    "For breast cancer, trastuzumab, neratinib, and tucatinib are used to treat HER",
    "We visited Greek temples, and saw the names of various Greek gods from Greek mythology, including ZEUS, POSEIDON, APOLLO, HADES and finally one for the brave HER",
    " ER",
    "Although HER2 amplification is usually considered in the context of breast cancer, it is clear that it can be amplified in other types of cancer also. In colorectal cancer, a subset of patients are found to have HER2 amplification as indicated by copy number increases in ER",
    "We visited the hospital, which is famous for all the gunshot wounds that come into the emergency room. We were told that our friend was being treated by a doctor in the ER",
    ]

TOP_K_NEXT = 10

base_context_size = int(base_cfg["context_length"])
dapt_context_size = int(dapt_model_cfg["context_length"])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for idx, prompt in enumerate(GPROMPTS, start=1):
    base_df = ta.top_next_tokens(base_model, prompt, base_context_size, top_k=TOP_K_NEXT)
    dapt_df = ta.top_next_tokens(dapt_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_df.add_prefix("base_"), dapt_df.add_prefix("dapt_")],
        axis=1,
    )

    print(f"PROMPT{idx:02d}:", prompt)
    print()
    print("Base vs DAPT top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")

# check ERBB2 predicted prompt specifically  
print("BB predicted next token analysis")
print(GPROMPTS[6])
BBprob_base = ta.next_token_probability(base_model, GPROMPTS[6], "BB", base_context_size)
BBprob_DAPT = ta.next_token_probability(dapt_model, GPROMPTS[6], "BB", dapt_context_size)
print(f"Base P('BB'): {BBprob_base:.6f}")
print(f"DAPT P('BB'): {BBprob_DAPT:.6f}")


PROMPT01:  dogs and

Base vs DAPT top next tokens:
   base_rank  base_tokenid   base_token  base_probability  dapt_rank  dapt_tokenid   dapt_token  dapt_probability
0          1           584     ' other'          0.042839          1           262       ' the'          0.030889
1          2           511     ' their'          0.037026          2           511     ' their'          0.025677
2          3          6844      ' dogs'          0.036724          3          9296    ' breast'          0.022798
3          4         11875      ' cats'          0.027796          4         10693      ' mice'          0.019679
4          5           257         ' a'          0.024118          5          1729       ' non'          0.016895
5          6           262       ' the'          0.023013          6         24906       ' HER'          0.014863
6          7          1751  ' children'          0.014539          7          1692     ' human'          0.012942
7          8         14260    ' horse

**RESULT**: Interesting result: A simple control prompt (PROMPT01) showed the effects of training, in that ' HER' and ' EG' showed up in the DAPT model, although at low probabilities (~ 1%). The control prompt (PROMPT02) did not change much as to the top most probable tokens, indicating that context was still overall important. PROMPT03 looked at the next most probable after ' HER', notably, '2' was absent in both top 10 listings. PROMPT04, aimed toward predicting '2' as the next token after ' HER', showed a large probability change for '2' with the DAPT model. PROMPT05, aimed toward producing 'O' (as in HERO), showed similar probabilities in both models, although in the DAPT model, '-' was the most probable token, probably showing the influence of training. PROMPT06 examined only ' ER' and notably, 'BB' was not in either top 10. PROMPT07, which should produce 'BB' as the most probable token (as in ERBB2, so ' ER' + 'BB' + '2'), showed a dramatic difference, with the base model not having 'BB' in the top 10 while it was easily the top token in the DAPT model. PROMPT08 was designed as a control with ' ER' in this context being an acronym for Emergency Room, so the next token could be many things. The top next predicted tokens for this prompt made sense, and notably, the DAPT model did have the presence of 'BB' in the top 10 (but at ~4%), again indicating the strength of some of the shifts with training. In total, these results demonstrate that next token probabilities are clearly shifting, consistent with the loss decrease in the DAPT model in the validation set. Combined with the results from the token embedding analyses, this points toward model alterations beyond simply creating similarity between some frequent tokens at the token embedding layer.  

### 5b. Which parameters and model blocks changed most after DAPT?

In [12]:
num_to_print=200
param_df = wd.compare_models(base_model, dapt_model, max_changed_tensors=num_to_print)
block_df = wd.summarize_by_gpt2_block(param_df)
block_proper_df = wd.summarize_by_block_proper(param_df, base_model, dapt_model)
submodule_df = wd.summarize_by_submodule(param_df)

print("Top changed individual tensors:")
print(param_df.head(num_to_print).to_string(index=False))

print("\nChanged blocks (simple summary):")
print(block_df.to_string(index=False))

print("\nChanged blocks (proper aggregated relative L2):")
print(block_proper_df.to_string(index=False))

print("\nChanged submodules:")
print(submodule_df.to_string(index=False))


Top changed individual tensors:
                             name    numel   rel_l2  delta_norm  mean_abs_delta  max_abs_delta  cosine_similarity  base_norm  dapt_norm
         trf_blocks.0.norm1.shift      768 0.242853    0.244849        0.006692       0.049637           0.971098   1.008217   0.933885
        trf_blocks.11.norm2.shift      768 0.241008    0.268515        0.007573       0.042380           0.970836   1.114131   1.109112
                   tok_emb.weight 38597376 0.239102  212.916107        0.026729       0.251837           0.976101 890.481873 878.440002
        trf_blocks.10.norm2.shift      768 0.160957    0.221184        0.006282       0.043843           0.986970   1.374178   1.350516
         trf_blocks.4.norm2.shift      768 0.134788    0.100420        0.002811       0.018054           0.990915   0.745021   0.744957
         trf_blocks.9.norm2.shift      768 0.134301    0.171678        0.004884       0.022267           0.990950   1.278306   1.261036
         trf_blo

## 6. Base model token embedding replaced with DAPT token embedding  

In [13]:
base_encode_plusothers_dapt_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_encode_plusothers_dapt_model, base_params)
base_encode_plusothers_dapt_model.out_head.weight = base_encode_plusothers_dapt_model.tok_emb.weight

if base_encode_plusothers_dapt_model.tok_emb.weight.shape != dapt_model.tok_emb.weight.shape:
    raise ValueError(
        f"tok_emb shape mismatch: {tuple(base_encode_plusothers_dapt_model.tok_emb.weight.shape)} vs {tuple(dapt_model.tok_emb.weight.shape)}"
    )

if base_encode_plusothers_dapt_model.pos_emb.weight.shape != dapt_model.pos_emb.weight.shape:
    raise ValueError(
        f"pos_emb shape mismatch: {tuple(base_encode_plusothers_dapt_model.pos_emb.weight.shape)} vs {tuple(dapt_model.pos_emb.weight.shape)}"
    )

with torch.no_grad():
    base_encode_plusothers_dapt_model.tok_emb.weight.copy_(dapt_model.tok_emb.weight)
    base_encode_plusothers_dapt_model.pos_emb.weight.copy_(dapt_model.pos_emb.weight)
    base_encode_plusothers_dapt_model.trf_blocks[0].norm1.shift.copy_(dapt_model.trf_blocks[0].norm1.shift)
    base_encode_plusothers_dapt_model.trf_blocks[4].norm2.shift.copy_(dapt_model.trf_blocks[4].norm2.shift)
    base_encode_plusothers_dapt_model.trf_blocks[9].norm2.shift.copy_(dapt_model.trf_blocks[9].norm2.shift)
    base_encode_plusothers_dapt_model.trf_blocks[10].norm2.shift.copy_(dapt_model.trf_blocks[10].norm2.shift)
    base_encode_plusothers_dapt_model.trf_blocks[11].norm2.shift.copy_(dapt_model.trf_blocks[11].norm2.shift)

base_encode_plusothers_dapt_model.eval()
print("Loaded base_encode_plusothers_dapt_model as a duplicate of base_model")
print("Replaced base_encode_plusothers_dapt_model tok_emb with DAPT tok_emb")
print("Replaced base_encode_plusothers_dapt_model pos_emb with DAPT pos_emb")
print("Replaced base_encode_plusothers_dapt_model trf_blocks.0.norm1.shift with DAPT value")
print("Replaced base_encode_plusothers_dapt_model trf_blocks.4.norm2.shift with DAPT value")
print("Replaced base_encode_plusothers_dapt_model trf_blocks.9.norm2.shift with DAPT value")
print("Replaced base_encode_plusothers_dapt_model trf_blocks.10.norm2.shift with DAPT value")
print("Replaced base_encode_plusothers_dapt_model trf_blocks.11.norm2.shift with DAPT value")
print("base_encode_plusothers_dapt_model tok_emb shape:", tuple(base_encode_plusothers_dapt_model.tok_emb.weight.shape))

# create base_encode_pos_trf0_11_model
base_encode_pos_trf0_11_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_encode_pos_trf0_11_model, base_params)
base_encode_pos_trf0_11_model.out_head.weight = base_encode_pos_trf0_11_model.tok_emb.weight

base_encode_pos_trf0_11_sd = base_encode_pos_trf0_11_model.state_dict()
dapt_sd = dapt_model.state_dict()
prefixes_to_replace = ["tok_emb.", "pos_emb.", "trf_blocks.0.", "trf_blocks.11."]
replaced_tensor_names = []

with torch.no_grad():
    for tensor_name, target_tensor in base_encode_pos_trf0_11_sd.items():
        if not any(tensor_name.startswith(prefix) for prefix in prefixes_to_replace):
            continue
        if tensor_name not in dapt_sd:
            raise KeyError(f"{tensor_name} missing from dapt_model state_dict")
        source_tensor = dapt_sd[tensor_name]
        if target_tensor.shape != source_tensor.shape:
            raise ValueError(
                f"shape mismatch for {tensor_name}: {tuple(target_tensor.shape)} vs {tuple(source_tensor.shape)}"
            )
        target_tensor.copy_(source_tensor)
        replaced_tensor_names.append(tensor_name)

if base_encode_pos_trf0_11_model.out_head.weight is not base_encode_pos_trf0_11_model.tok_emb.weight:
    raise ValueError("base_encode_pos_trf0_11_model lost weight tying between tok_emb and out_head")

base_encode_pos_trf0_11_model.eval()
print("Loaded base_encode_pos_trf0_11_model as a duplicate of base_model")
print("Replaced tok_emb, pos_emb, trf_blocks.0.*, and trf_blocks.11.* with DAPT values")
print("Number of tensors replaced in base_encode_pos_trf0_11_model:", len(replaced_tensor_names))
print("base_encode_pos_trf0_11_model tok_emb shape:", tuple(base_encode_pos_trf0_11_model.tok_emb.weight.shape))

# create base_encode_pos_trf5_6_model
base_encode_pos_trf5_6_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_encode_pos_trf5_6_model, base_params)
base_encode_pos_trf5_6_model.out_head.weight = base_encode_pos_trf5_6_model.tok_emb.weight

base_encode_pos_trf5_6_sd = base_encode_pos_trf5_6_model.state_dict()
prefixes_to_replace = ["tok_emb.", "pos_emb.", "trf_blocks.5.", "trf_blocks.6."]
replaced_tensor_names = []

with torch.no_grad():
    for tensor_name, target_tensor in base_encode_pos_trf5_6_sd.items():
        if not any(tensor_name.startswith(prefix) for prefix in prefixes_to_replace):
            continue
        if tensor_name not in dapt_sd:
            raise KeyError(f"{tensor_name} missing from dapt_model state_dict")
        source_tensor = dapt_sd[tensor_name]
        if target_tensor.shape != source_tensor.shape:
            raise ValueError(
                f"shape mismatch for {tensor_name}: {tuple(target_tensor.shape)} vs {tuple(source_tensor.shape)}"
            )
        target_tensor.copy_(source_tensor)
        replaced_tensor_names.append(tensor_name)

if base_encode_pos_trf5_6_model.out_head.weight is not base_encode_pos_trf5_6_model.tok_emb.weight:
    raise ValueError("base_encode_pos_trf5_6_model lost weight tying between tok_emb and out_head")

base_encode_pos_trf5_6_model.eval()
print("Loaded base_encode_pos_trf5_6_model as a duplicate of base_model")
print("Replaced tok_emb, pos_emb, trf_blocks.5.*, and trf_blocks.6.* with DAPT values")
print("Number of tensors replaced in base_encode_pos_trf5_6_model:", len(replaced_tensor_names))
print("base_encode_pos_trf5_6_model tok_emb shape:", tuple(base_encode_pos_trf5_6_model.tok_emb.weight.shape))

# create base_encode_dapt model
base_encode_dapt_model = gpt2.setup_model(base_cfg).to(device)
training_utils.load_weights_into_gpt(base_encode_dapt_model, base_params)
base_encode_dapt_model.out_head.weight = base_encode_dapt_model.tok_emb.weight

if base_encode_dapt_model.tok_emb.weight.shape != dapt_model.tok_emb.weight.shape:
    raise ValueError(
        f"tok_emb shape mismatch: {tuple(base_encode_dapt_model.tok_emb.weight.shape)} vs {tuple(dapt_model.tok_emb.weight.shape)}"
    )

with torch.no_grad():
    base_encode_dapt_model.tok_emb.weight.copy_(dapt_model.tok_emb.weight)

base_encode_dapt_model.eval()
print("Loaded base_encode_dapt_model as a duplicate of base_model")
print("Replaced base_encode_dapt_model tok_emb with DAPT tok_emb")
print("base_encode_dapt_model tok_emb shape:", tuple(base_encode_dapt_model.tok_emb.weight.shape))

# create dapt_encode_base model
dapt_encode_base_model = gpt2.setup_model(dapt_model_cfg).to(device)
dapt_encode_base_model.load_state_dict(dapt_model.state_dict(), strict=True)

if dapt_encode_base_model.tok_emb.weight.shape != base_model.tok_emb.weight.shape:
    raise ValueError(
        f"tok_emb shape mismatch: {tuple(dapt_encode_base_model.tok_emb.weight.shape)} vs {tuple(base_model.tok_emb.weight.shape)}"
    )

with torch.no_grad():
    dapt_encode_base_model.tok_emb.weight.copy_(base_model.tok_emb.weight)

if dapt_encode_base_model.out_head.weight is not dapt_encode_base_model.tok_emb.weight:
    raise ValueError("dapt_encode_base_model lost weight tying between tok_emb and out_head")

dapt_encode_base_model.eval()
print("Loaded dapt_encode_base_model as a duplicate of dapt_model")
print("Replaced dapt_encode_base_model tok_emb with base tok_emb")
print("dapt_encode_base_model tok_emb shape:", tuple(dapt_encode_base_model.tok_emb.weight.shape))


Loaded base_encode_plusothers_dapt_model as a duplicate of base_model
Replaced base_encode_plusothers_dapt_model tok_emb with DAPT tok_emb
Replaced base_encode_plusothers_dapt_model pos_emb with DAPT pos_emb
Replaced base_encode_plusothers_dapt_model trf_blocks.0.norm1.shift with DAPT value
Replaced base_encode_plusothers_dapt_model trf_blocks.4.norm2.shift with DAPT value
Replaced base_encode_plusothers_dapt_model trf_blocks.9.norm2.shift with DAPT value
Replaced base_encode_plusothers_dapt_model trf_blocks.10.norm2.shift with DAPT value
Replaced base_encode_plusothers_dapt_model trf_blocks.11.norm2.shift with DAPT value
base_encode_plusothers_dapt_model tok_emb shape: (50257, 768)
Loaded base_encode_pos_trf0_11_model as a duplicate of base_model
Replaced tok_emb, pos_emb, trf_blocks.0.*, and trf_blocks.11.* with DAPT values
Number of tensors replaced in base_encode_pos_trf0_11_model: 36
base_encode_pos_trf0_11_model tok_emb shape: (50257, 768)
Loaded base_encode_pos_trf5_6_model as a

In [14]:
TOP_K_NEXT = 30

base_alt_context_size = int(base_cfg["context_length"])
dapt_context_size = int(dapt_model_cfg["context_length"])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for idx, prompt in enumerate(GPROMPTS, start=1):
    base_alt_df = ta.top_next_tokens(base_encode_plusothers_dapt_model, prompt, base_alt_context_size, top_k=TOP_K_NEXT)
    dapt_df = ta.top_next_tokens(dapt_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_alt_df.add_prefix("base_alt_"), dapt_df.add_prefix("dapt_")],
        axis=1,
    )

    print(f"PROMPT{idx:02d}:", prompt)
    print()
    print("Base_alt vs DAPT top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")

# check ERBB2 predicted prompt specifically  
print("BB predicted next token analysis")
print(GPROMPTS[6])
BBprob_base_alt = ta.next_token_probability(base_encode_plusothers_dapt_model, GPROMPTS[6], "BB", base_alt_context_size)
BBprob_DAPT = ta.next_token_probability(dapt_model, GPROMPTS[6], "BB", dapt_context_size)
print(f"Base_alt P('BB'): {BBprob_base_alt:.6f}")
print(f"DAPT P('BB'): {BBprob_DAPT:.6f}")


PROMPT01:  dogs and

Base_alt vs DAPT top next tokens:
    base_alt_rank  base_alt_tokenid base_alt_token  base_alt_probability  dapt_rank  dapt_tokenid   dapt_token  dapt_probability
0               1              1309         ' let'              0.077822          1           262       ' the'          0.030889
1               2               511       ' their'              0.071032          2           511     ' their'          0.025677
2               3               262         ' the'              0.036855          3          9296    ' breast'          0.022798
3               4               584       ' other'              0.029257          4         10693      ' mice'          0.019679
4               5               257           ' a'              0.026493          5          1729       ' non'          0.016895
5               6               772        ' even'              0.018381          6         24906       ' HER'          0.014863
6               7              6844       

### dapt_encode_base_model vs base_model prompt-based evaluation

In [15]:
TOP_K_NEXT = 10

# context sizes are the same for all the models, hence this is not a big deal
base_context_size = int(base_cfg["context_length"])
dapt_context_size = base_context_size

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for idx, prompt in enumerate(GPROMPTS, start=1):
    base_df = ta.top_next_tokens(base_model, prompt, base_alt_context_size, top_k=TOP_K_NEXT)
    dapt_alt_df = ta.top_next_tokens(dapt_encode_base_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_df.add_prefix("base_"), dapt_alt_df.add_prefix("dapt_alt_")],
        axis=1,
    )

    print(f"PROMPT{idx:02d}:", prompt)
    print()
    print("Base vs DAPT_alt top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")

# check ERBB2 predicted prompt specifically  
print("BB predicted next token analysis")
print(GPROMPTS[6])
BBprob_base = ta.next_token_probability(base_model, GPROMPTS[6], "BB", base_context_size)
BBprob_DAPT_alt = ta.next_token_probability(dapt_encode_base_model, GPROMPTS[6], "BB", dapt_context_size)
print(f"Base P('BB'): {BBprob_base:.6f}")
print(f"DAPT_alt P('BB'): {BBprob_DAPT_alt:.6f}")


PROMPT01:  dogs and

Base vs DAPT_alt top next tokens:
   base_rank  base_tokenid   base_token  base_probability  dapt_alt_rank  dapt_alt_tokenid dapt_alt_token  dapt_alt_probability
0          1           584     ' other'          0.042839              1             10693        ' mice'              0.016786
1          2           511     ' their'          0.037026              2               511       ' their'              0.008549
2          3          6844      ' dogs'          0.036724              3             13623        ' rats'              0.008430
3          4         11875      ' cats'          0.027796              4             33043     ' rabbits'              0.006587
4          5           257         ' a'          0.024118              5               262         ' the'              0.005525
5          6           262       ' the'          0.023013              6              6844        ' dogs'              0.004206
6          7          1751  ' children'          

## 7. Full validation-set loss evaluation across all seven models

In [16]:
from llm_from_scratch.dataloader import dataloader

VAL_ABSTRACTS_PATH = DATA_DIR / "pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_val_abstracts.txt"

if not VAL_ABSTRACTS_PATH.exists():
    raise FileNotFoundError(f"Validation abstracts file not found: {VAL_ABSTRACTS_PATH}")

validation_set_fraction = 0.025

if not (0 < validation_set_fraction <= 1):
    raise ValueError(f"validation_set_fraction must be > 0 and <= 1, got {validation_set_fraction}")

val_text_full = dataloader.load_file(VAL_ABSTRACTS_PATH)
val_text_num_chars = max(1, int(len(val_text_full) * validation_set_fraction))
val_text = val_text_full[:val_text_num_chars]

# Match the full-set evaluation style used at the end of training: no shuffle, no dropped last batch,
# and loss computed across the entire validation loader.
val_batch_size = int(checkpoint.get("run_config", {}).get("batch_size", 2))
val_stride = int(checkpoint.get("run_config", {}).get("stride", dapt_model_cfg["context_length"]))
val_context_length = int(dapt_model_cfg["context_length"])

full_val_loader = dataloader.create_dataloader_v1(
    val_text,
    batch_size=val_batch_size,
    max_length=val_context_length,
    stride=val_stride,
    shuffle=False,
    drop_last=True,
    num_workers=0,
)

print("Validation file:", VAL_ABSTRACTS_PATH)
print("Validation set fraction:", validation_set_fraction)
print("Validation characters used:", len(val_text), "of", len(val_text_full))
print("Validation batch size:", val_batch_size)
print("Validation context length:", val_context_length)
print("Validation stride:", val_stride)
print("Number of validation batches:", len(full_val_loader))

def eval_full_val_set(model, val_loader, device):
    was_training = model.training
    model.eval()
    with torch.no_grad():
        val_loss = training_utils.calc_loss_loader(val_loader, model, device, num_batches=None)
    if was_training:
        model.train()
    return val_loss

val_loss_rows = []
models_to_eval = [
    ("base_model", base_model),
    ("base_encode_plusothers_dapt_model", base_encode_plusothers_dapt_model),
    ("base_encode_pos_trf0_11_model", base_encode_pos_trf0_11_model),
    ("base_encode_pos_trf5_6_model", base_encode_pos_trf5_6_model),
    ("base_encode_dapt_model", base_encode_dapt_model),
    ("dapt_model", dapt_model),
    ("dapt_encode_base_model", dapt_encode_base_model),
]

for model_name, model in models_to_eval:
    full_val_loss = eval_full_val_set(model, full_val_loader, device)
    print(f"{model_name} FULL val loss: {full_val_loss:.6f}")
    val_loss_rows.append({
        "model": model_name,
        "validation_set_fraction": validation_set_fraction,
        "full_val_loss": full_val_loss,
    })

val_loss_df = pd.DataFrame(val_loss_rows).sort_values("full_val_loss").reset_index(drop=True)
print()
print(val_loss_df.to_string(index=False))


Validation file: /home/markb/cloned-llm-2026MAR16/llm-from-scratch/data/pubmed_abstracts_2005to2025ONLY_ERBB2_ABSTRACTS_getv7_english_val_abstracts.txt
Validation set fraction: 0.025
Validation characters used: 72602 of 2904096
Validation batch size: 2
Validation context length: 1024
Validation stride: 1024
Number of validation batches: 8
base_model FULL val loss: 2.958093
base_encode_plusothers_dapt_model FULL val loss: 3.268645
base_encode_pos_trf0_11_model FULL val loss: 2.879578
base_encode_pos_trf5_6_model FULL val loss: 3.171551
base_encode_dapt_model FULL val loss: 3.303649
dapt_model FULL val loss: 2.596266
dapt_encode_base_model FULL val loss: 2.905456

                            model  validation_set_fraction  full_val_loss
                       dapt_model                    0.025       2.596266
    base_encode_pos_trf0_11_model                    0.025       2.879578
           dapt_encode_base_model                    0.025       2.905456
                       base_model